# IngestThis notebook mocks the ingestion pipeline for our proof of concept. Every organisation uses a different configuration for ingestion, but we can abstract this into two main tasks:1. Ingest documents from storage systems into cloud storage2. Replicate or mount this cloud storage into Unity Catalog VolumesThe goal here is simply to land the documents and capture their lineage via a clear save path. The `02_convert` Notebook covers the document processing.This notebook has been tested with serverless v3.

In [0]:
%pip install mlflow>3.1

Our ingestion is driven off a table of URLs. This could also be easily done via URIs from blob storage (e.g. adfss) or volume paths. We use a default table in the `assets` folder to drive our loads, saving it as a delta table after we are done processing.

In [0]:
from mlflow.models import ModelConfigimport pandas as pdfrom src.utils import get_spark

In [0]:
config = ModelConfig(development_config="config.yaml")CATALOG = config.get("data").get("catalog")SCHEMA = config.get("data").get("schema")RAW_DOCS_VOL = config.get("data").get("raw_docs_vol")PROCESSED_DOCS_VOL = config.get("data").get("processed_docs_vol")#spark = get_spark()#doc_df = pd.read_csv("./assets/forge_reports.csv")#doc_paths = spark.createDataFrame(doc_df)

We now use that driver table to download all the documents. We use a medallion architecture in volumes as well, landing in a bronze folder, processing into a silver folder, and serving from a gold folder.

In [0]:
# Ensure catalog, schema, and volumes are readyfrom databricks.sdk.service.catalog import VolumeTypefrom databricks.sdk import WorkspaceClientw = WorkspaceClient()try:    w.catalogs.create(name=CATALOG)except:    print(f"{CATALOG} catalog exists")try:    w.schemas.create(catalog_name=CATALOG, name=SCHEMA)except:    print(f"{SCHEMA} catalog exists")for vol_name in [RAW_DOCS_VOL, PROCESSED_DOCS_VOL]:    try:        w.volumes.create(            catalog_name=CATALOG,            schema_name=SCHEMA,            name=vol_name,            volume_type=VolumeType.MANAGED,        )    except:        print(f"{vol_name} volume exists")

We use a standard spark user defined function (UDF) to download the files and setup a driver table with the downloaded doc path. We use a spark UDF to download all the files in parallel.

In [0]:
import pyspark.sql.functions as Fimport pyspark.sql.types as Timport requestsfrom src.document.utils import sanitize_filenameRAW_DOC_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{RAW_DOCS_VOL}"@udf(T.StringType())def download_file(url):    file_name = sanitize_filename(url)    saved_file_path = f"{RAW_DOC_DIR}/{file_name}"    response = requests.get(url)    with open(saved_file_path, "wb") as file:        file.write(response.content)    return saved_file_path

In [0]:
doc_paths = doc_paths.withColumn(    "saved_file_path", download_file(F.col("download_link")))(    doc_paths.write.format("delta")    .mode("overwrite")    .option("mergeSchema", "true")    .saveAsTable(f"{CATALOG}.{SCHEMA}.documents"))

In [0]:
display(doc_paths)

In [0]:
%sqlcreate table devanshu_pandey.multimodal.documents as select * from shm.multimodal.documentslimit 0;

In [0]:
%sqlselect * from shm.multimodal.documents limit 1

In [0]:
spark.sql("""INSERT INTO devanshu_pandey.multimodal.documents (id, title, url, download_link, saved_file_path)VALUES (1, 'WikiHow Adobe Photoshop', 'https://www.wikihow.com/Use-Adobe-Photoshop', 'https://www.wikihow.com/Use-Adobe-Photoshop', '/Volumes/devanshu_pandey/multimodal/raw_docs/wikihow_adobe_photoshop.pdf')""")